# Model Training

**1.1 Importing Data and Required Packages**
- Importing Pandas, Numpy,Matplotlib, Seaborn and Warnings Library

In [2]:
# Necessary import
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 


from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('data/creditcard.csv')

In [4]:
X = df.drop('Class', axis=1) # features (Time, Amount , V1-28)
y = df["Class"] # label(fraud or non fraud)

In [5]:
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42, stratify = y)



In [6]:
print("Train fraud % :", y_train.mean())
print("Test fraud % :", y_test.mean())

Train fraud % : 0.001729245759178389
Test fraud % : 0.0017204452090867595


In [7]:
from sklearn.linear_model import LogisticRegression # imports Logistic Regression model because its simple to start with
from sklearn.preprocessing import StandardScaler # standardizes features to have mean 0 and standard deviation 1, enhancing model performance and accuracy.
scaler = StandardScaler() # create scalar object to normalise feature values
X_train_scaled = scaler.fit_transform(X_train) # learn mean and std from training data and scaling it
model = LogisticRegression(max_iter=1000, class_weight= "balanced") #create logistic model, allow more iterations & handle class imbalances
model.fit(X_train_scaled, y_train) # train model on training data
X_test_scaled = scaler.transform(X_test) #apply same scaling learned from training data to test data

model.fit(X_train_scaled, y_train) # Re-training the model on training data
y_pred = model.predict(X_test_scaled) # predict faud (0-1) on unseen test data
y_prob = model.predict_proba(X_test_scaled)[:,1] # gives frauds probabbibilty

from sklearn.metrics import classification_report, roc_auc_score #import tools to evaluate how good the model is, classification_report gives precision, recall, F1, 
#roc_auc_score gives overall ranking ability 

print(classification_report(y_test, y_pred)) # prints precision, recall and F1 score for fraud and non-fraud
print("ROC_AUC: ", roc_auc_score(y_test, y_prob)) # prints ROC-AUC Score ( how well model separates classes overall)

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962

ROC_AUC:  0.9720834996210077


# BaseLine Logistic Regression Results
- There were 98 actual fraud cases in test dataset
- Out of all frauds, we correctly caught about 90 (92% recall displayed from output above which 0.92 converted to percentage to get 92%).
- Only 6% of flag transaction are actually fraud (low precision) (from output above, 0.06 converted to 6 %)
- This means model catches most fraud cases but produces many false alarms(means that model says this transaction is fraud but its actually a normal transaction)
- ROC-AUC Score is 0.97, showing that the model is good at separating fraud from normal transactions overall

# Threshhold Tuning, Trying Different probability cutoffs

In [ ]:
threshold = 0.3
y_pred_03 = (y_prob >= threshold).astype(int) # Checks for each probabibility, if it's >= 0.3, mark true else false, 
# then convert True/ False in 0/1

print("Threshhold: 0.3")
print(classification_report(y_test, y_pred_03))# print precision/recall/F1 by comparing real labels(of y_test)
# with new predictions(y_pred_03)


In [10]:
threshold = 0.8
y_pred_03 = (y_prob >= threshold).astype(int)

print("Threshold: 0.8")
print(classification_report(y_test, y_pred_03))

Threshold: 0.8
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     56864
           1       0.16      0.89      0.27        98

    accuracy                           0.99     56962
   macro avg       0.58      0.94      0.63     56962
weighted avg       1.00      0.99      0.99     56962



## Threshold Tuning Results

### Threshold = 0.3

- Fraud recall remained high (92%), meaning most fraud cases were detected.
- Fraud precision dropped to 3%, meaning most flagged transactions were actually normal.
- Lowering the threshold increases fraud detection but causes many false alarms.

### Interpretation

- Lower threshold makes the model more aggressive.
- It catches more fraud but wrongly flags many normal transactions.
- This shows the tradeoff between precision and recall.

